# Stage 3 concat576 LoRA — official benchmark evaluation

Full pipeline: **576 global tokens + K×16 region tokens + LoRA**.

POPE, VQAv2 test-dev, MMBench, and SEED do **not** ship question-specific boxes.
Every sample uses automatic proposals via `run_combined_inference`:

1. **spaCy** — extract noun phrases from the question (heuristic fallback if spaCy unavailable)
2. **RAM++** — image-level tags
3. **Union** — `spaCy nouns ∪ RAM tags` (`box_source="hybrid"`)
4. **Grounding DINO** — localize each tag → up to K boxes (default 20)
5. **concat576 forward** — `[576 globals] + [K×16 regions] + question`

Set `BOX_SOURCE = "question"` for spaCy nouns only (no RAM). GQA excluded (train/eval leakage).

| Benchmark | Scoring |
|-----------|---------|
| VQAv2 test-dev | EvalAI submission (~447k questions) |
| POPE | macro F1 over random / popular / adversarial |
| MMBench EN dev | local accuracy + VLMEvalKit TSV export |
| SEED-Bench image | dims 1–9 MCQ accuracy |

Helpers: `evaluation.py` · JSONL resume supported.

In [ ]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


## 0. Environment

In [ ]:
import os
from pathlib import Path

# Must run before importing transformers / huggingface_hub.
from reva.config import configure_hf_cache

HF_CACHE = configure_hf_cache(os.environ["REVA_HF_CACHE_ROOT"])
print("HF cache root:", HF_CACHE)
print("HF hub cache:", HF_CACHE / "hub")
print("Free in /tmp:", f"{os.statvfs('/tmp').f_bavail * os.statvfs('/tmp').f_frsize / 1e9:.0f}G")

os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
os.environ.setdefault("USE_TORCH", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    free_gb, total_gb = torch.cuda.mem_get_info()
    print(f"GPU mem free: {free_gb / 1e9:.1f} / {total_gb / 1e9:.1f} GB")

## 1. Paths and run controls

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(".").resolve()
DATA_ROOT = Path(os.environ.get("REVA_DATA_ROOT") or os.path.expanduser("~/reva-data"))
PROJECTION_B = PROJECT_DIR / "projection_b_best_weights.pt"
PROJECTION_A = PROJECT_DIR / "projection_a_curriculum_best_weights.pt"

# Point this at your completed Stage 3 run's best_lora directory.
LORA = Path("/home/jovyan/teaching_material/stage3_lora")
OUTPUT_DIR = DATA_ROOT / "eval_results/concat576_full_pipeline"

MAX_BOXES = 20
BOX_SOURCE = "hybrid"       # hybrid | ram → spaCy∪RAM; question → spaCy only
LOAD_RAM = True             # required for hybrid/ram
LIMIT = None                # use 10 for a smoke test
RESUME = True               # False deletes selected benchmark's old JSONL

RUN_VQAV2 = False
RUN_POPE = True
RUN_MMBENCH = False
RUN_SEED_IMAGE = False

for path in (PROJECTION_B, PROJECTION_A, LORA):
    assert path.exists(), path
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outputs:", OUTPUT_DIR)
print("Box proposal:", BOX_SOURCE, "| max boxes:", MAX_BOXES)

## 2. Load concat576 full-pipeline stack

In [ ]:
from reva.evaluation import (
    evaluate_mmbench_full,
    evaluate_pope_full,
    evaluate_seed_image_full,
    evaluate_vqav2_testdev_full,
    load_stage3_eval_stack,
)

stack = load_stage3_eval_stack(
    projection_b_path=PROJECTION_B,
    projection_a_path=PROJECTION_A,
    lora_path=LORA,
    max_boxes=MAX_BOXES,
    box_source=BOX_SOURCE,
    load_ram=LOAD_RAM,
)
print("Loaded on", stack.config.device, "|", stack.config.compute_dtype)
print("Visual prefix: 576 globals + up to", MAX_BOXES, "× 16 region tokens")
print("Inference path: run_combined_inference (spaCy∪RAM → DINO → concat576)")

## 3. Run selected benchmarks

In [ ]:
summaries = {}

if RUN_VQAV2:
    summaries["vqav2_testdev"] = evaluate_vqav2_testdev_full(
        stack, vqav2_root=DATA_ROOT / "vqav2", output_dir=OUTPUT_DIR,
        limit=LIMIT, resume=RESUME,
    )

if RUN_POPE:
    summaries["pope"] = evaluate_pope_full(
        stack, pope_root=DATA_ROOT / "pope",
        coco_val2014_dir=DATA_ROOT / "coco/val2014", output_dir=OUTPUT_DIR,
        limit=LIMIT, resume=RESUME,
    )

if RUN_MMBENCH:
    summaries["mmbench"] = evaluate_mmbench_full(
        stack, mmbench_root=DATA_ROOT / "mmbench", output_dir=OUTPUT_DIR,
        split="MMBench_DEV_EN", limit=LIMIT, resume=RESUME,
    )

if RUN_SEED_IMAGE:
    summaries["seed_image"] = evaluate_seed_image_full(
        stack, seed_root=DATA_ROOT / "seed_bench", output_dir=OUTPUT_DIR,
        limit=LIMIT, resume=RESUME,
    )

## 4. Report and save combined summary

In [ ]:
import json

for name, result in summaries.items():
    if "accuracy_pct" in result:
        print(f"{name:16s}: {result['accuracy_pct']:.2f}% (n={result['n']})")
    elif "macro_f1_pct" in result:
        print(f"{name:16s}: F1={result['macro_f1_pct']:.2f}% (n={result['n']})")
    elif "local_accuracy_pct" in result:
        print(f"{name:16s}: local={result['local_accuracy_pct']:.2f}% (canonical via VLMEvalKit)")
    else:
        print(f"{name:16s}: {result['n']} predictions; external scoring required")

combined_path = OUTPUT_DIR / "all_summaries.json"
payload = {
    "experiment": "stage3_concat576_lora",
    "lora_path": str(LORA),
    "projection_b_path": str(PROJECTION_B),
    "projection_a_path": str(PROJECTION_A),
    "box_source": BOX_SOURCE,
    "max_boxes": MAX_BOXES,
    "visual_input": "576 globals + K×16 regions (spaCy∪RAM → DINO)",
    "summaries": summaries,
}
with open(combined_path, "w") as f:
    json.dump(payload, f, indent=2)
print("Saved:", combined_path)
print("VQAv2 submission:", OUTPUT_DIR / "vqav2_testdev_submission.json")

## 5. Read POPE scores (after POPE finishes)

Scores are saved to **`pope_summary.json`** under `OUTPUT_DIR`.

Report **`macro_f1_pct`** as POPE F1 in your paper table.

In [ ]:
import json
from pathlib import Path

pope_path = OUTPUT_DIR / "pope_summary.json"
if not pope_path.exists():
    print("Run POPE first (section 3). Expected:", pope_path)
else:
    pope = json.loads(pope_path.read_text())
    print(f"POPE macro F1: {pope['macro_f1_pct']:.2f}%")
    print(f"POPE macro accuracy: {pope['macro_accuracy_pct']:.2f}%")
    print(f"Total n: {pope['n']}")
    print()
    for split, m in pope["splits"].items():
        print(
            f"  {split:12s}: F1={m['f1_pct']:.2f}%  "
            f"acc={m['accuracy_pct']:.2f}%  "
            f"prec={m['precision_pct']:.2f}%  "
            f"rec={m['recall_pct']:.2f}%  "
            f"invalid={m['invalid']}"
        )
    print()
    print("Per-split JSONL:")
    for split, m in pope["splits"].items():
        print(f"  {split}: {m['predictions_path']}")